## Data Processing in Python Final Project

### Authors: Matyáš Tvrz, Jonathan Eugenio Gaeta

### To Do List:

1) download data from bezrealitky.cz - DONE and reality.idnes.cz, and from sreality.cz on other cities - DONE
2) create heatmap based on longitude and latitude - DONE
3) add property popups to map - DONE
4) download info on airbnb prices
5) conduct a simple analysis of rental price determinants

In [ ]:
# import packages

import json
import pandas as pd
import os
import requests 
import pandas as pd 
import time
import re 
import random 
import folium
from folium.plugins import HeatMap, MarkerCluster, GroupedLayerControl
import math

### sreality.cz dataset

In [ ]:
# get sreality df from last request
df_sreality = pd.read_json("df_sreality.json")
df_sreality = pd.DataFrame(df_sreality)

In [ ]:
# or get newest sreality df, takes about 6 minutes
from function_scripts import request_sreality_all
df_sreality = request_sreality_all() 
df_sreality.to_json("df_sreality.json", orient="records")

In [ ]:
# get square meters (area) and flat type from name
from function_scripts import name_to_area
df_sreality['area'] = df_sreality.name.apply(name_to_area)
df_sreality['flat_type'] = df_sreality.name.apply(lambda x: x.split()[2])

# get link to listing and thumbnail image
from function_scripts import get_link_and_image
df_sreality = get_link_and_image(df_sreality)

display(df_sreality)

In [ ]:
# clean sreality dataset
df_sreality_clean = df_sreality[['locality', 'price', 'flat_type','area','gps','hash_id', 'url','image']].copy()

# get latitude and logitude, manually adjusted based on results
df_sreality_clean[['lat', 'lon']] = df_sreality_clean.gps.apply(lambda x: pd.Series({'lat': x['lat']+ 0.008, 'lon': x['lon']-0.008}))
df_sreality_clean = df_sreality_clean.drop(columns = ["gps", "hash_id"])

display(df_sreality_clean)


### bezrealitky.cz dataset

In [17]:
# get bezrealitky df from last request
df_bezrealitky = pd.read_json("df_bezrealitky.json")
df_bezrealitky = pd.DataFrame(df_bezrealitky)

In [ ]:
# or get newest bezrealitky df, takes about 10 minutes
from function_scripts import request_bezrealitky
bezrealitky_url = "https://www.bezrealitky.cz/vyhledat?estateType=BYT&location=exact&offerType=PRONAJEM&osm_value=%C4%8Cesko&regionOsmIds=R51684&currency=CZK"
df_bezrealitky = request_bezrealitky(bezrealitky_url, 171)
df_bezrealitky.to_json("df_bezrealitky.json", orient="records")

In [18]:
# convert flat_type to standard nomenclature
mapping = {
    'DISP_1_KK': '1+kk',
    'DISP_2_KK': '2+kk',
    'DISP_3_KK': '3+kk',
    'DISP_4_KK': '4+kk',
    'DISP_1_1': '1+1',
    'DISP_2_1': '2+1',
    'DISP_3_1': '3+1',
    'DISP_4_1': '4+1',
    'DISP_5_1': '5+1',
    'DISP_7_1': '7+1',
    'GARSONIERA': '1+kk',
    'OSTATNI': 'atypické',
    'UNDEFINED': 'atypické',
}

df_bezrealitky['flat_type'] = df_bezrealitky['flat_type'].map(mapping)
df_bezrealitky_clean = df_bezrealitky

display(df_bezrealitky_clean)

,locality,price,flat_type,area,url,image,lat,lon
0,"náměstí Jiřího z Poděbrad, Nový Knín - Nový Kn...",14000,2+kk,35,https://www.bezrealitky.cz/nemovitosti-byty-do...,https://api.bezrealitky.cz/media/cache/record_...,49.787301,14.293177
1,"Hřímalého, Plzeň - Plzeň, Plzeňský kraj",12000,1+1,43,https://www.bezrealitky.cz/nemovitosti-byty-do...,https://api.bezrealitky.cz/media/cache/record_...,49.737219,13.369286
2,"Na Celné, Praha - Smíchov",21900,1+1,35,https://www.bezrealitky.cz/nemovitosti-byty-do...,https://api.bezrealitky.cz/media/cache/record_...,50.070907,14.409744
3,"Milíčova, Praha - Žižkov",18500,1+kk,29,https://www.bezrealitky.cz/nemovitosti-byty-do...,https://api.bezrealitky.cz/media/cache/record_...,50.084999,14.451252
4,"Poděbradská, Praha - Hloubětín",21503,2+1,52,https://www.bezrealitky.cz/nemovitosti-byty-do...,https://api.bezrealitky.cz/media/cache/record_...,50.106896,14.541320
...,...,...,...,...,...,...,...,...
2557,Beim Farenland 23 Hamburg Farmsen-Berne Hambur...,1249000,atypické,200,https://www.bezrealitky.cz/nemovitosti-byty-do...,https://api.bezrealitky.cz/media/cache/record_...,53.633820,10.136530
2558,"H.-E.-Busse Str. 4, , Bádensko-Württembersko",940,3+kk,90,https://www.bezrealitky.cz/nemovitosti-byty-do...,https://api.bezrealitky.cz/media/cache/record_...,47.847930,9.008880
2559,"Eynattener Str., , Severní Porýní-Vestfálsko",750,2+1,48,https://www.bezrealitky.cz/nemovitosti-byty-do...,https://api.bezrealitky.cz/media/cache/record_...,50.763230,6.084700
2560,Schroeterstraße 22 Dresden Prohlis Sachsen 01237,40,atypické,0,https://www.bezrealitky.cz/nemovitosti-byty-do...,https://api.bezrealitky.cz/media/cache/record_...,51.013220,13.783890


### Pooling datasets

In [ ]:
df_all = pd.concat([df_sreality_clean, df_bezrealitky_clean], ignore_index=True)

display(df_all)

In [ ]:
df_heatmap = df_all[['lat', 'lon', 'price']].copy()
df_sreality_property = df_sreality_clean[['lat', 'lon', 'price','locality', 'flat_type', 'area', 'url', 'image']].copy()
df_bezrealitky_property = df_bezrealitky_clean[['lat', 'lon', 'price','locality', 'flat_type', 'area', 'url', 'image']].copy()

### Heatmap

In [ ]:
# base map, centered on CZ
m = folium.Map(location=(49.75, 15.40), zoom_start = 8)

heat_data = df_heatmap[['lat', 'lon', 'price']].values.tolist()

# heatmap layer
heat_layer = folium.FeatureGroup(name="Heat Map", show=True)

HeatMap(
    heat_data,
    min_opacity=0.4,
    blur=18,
    radius=15
).add_to(heat_layer)

heat_layer.add_to(m)

# realty website indicator
df_sreality_property['source'] = 'Sreality'
df_bezrealitky_property['source'] = 'Bezrealitky'

# sreality property popups layer
sreality_layer = folium.FeatureGroup(
    name="Sreality Properties",
    show=False
)

sreality_cluster = MarkerCluster().add_to(sreality_layer)

# create popups with variables from df_sreality_property
for _, row in df_sreality_property.iterrows():

    popup_html = f"""
    <b>{row['price']:,} CZK</b><br>
    <b>Source:</b> {row['source']}<br>
    {row['locality']}<br>
    {row['flat_type']}<br>
    {row['area']} m²<br><br>
    <img src="{row['image']}" width="200"><br>
    <a href="{row['url']}" target="_blank">Open listing</a>
    """

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        fill=True,
        popup=folium.Popup(popup_html, max_width=250)
    ).add_to(sreality_cluster)

sreality_layer.add_to(m)

# bezrealitky property popups layer
bezrealitky_layer = folium.FeatureGroup(
    name="Bezrealitky Properties",
    show=False
)

bezrealitky_cluster = MarkerCluster().add_to(bezrealitky_layer)

# create popups with variables from df_bezrealitky_property
for _, row in df_bezrealitky_property.iterrows():

    popup_html = f"""
    <b>{row['price']:,} CZK</b><br>
    <b>Source:</b> {row['source']}<br>
    {row['locality']}<br>
    {row['flat_type']}<br>
    {row['area']} m²<br><br>
    <img src="{row['image']}" width="200"><br>
    <a href="{row['url']}" target="_blank">Open listing</a>
    """

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        fill=True,
        popup=folium.Popup(popup_html, max_width=250)
    ).add_to(bezrealitky_cluster)

bezrealitky_layer.add_to(m)


# all datasets merged property popups
all_property = pd.concat(
    [df_sreality_property, df_bezrealitky_property],
    ignore_index=True
)

all_layer = folium.FeatureGroup(
    name="All Properties",
    show=True
)

all_cluster = MarkerCluster().add_to(all_layer)

# create popups with variables from merged dataset
for _, row in all_property.iterrows():

    popup_html = f"""
    <b>{row['price']:,} CZK</b><br>
    <b>Source:</b> {row['source']}<br>
    {row['locality']}<br>
    {row['flat_type']}<br>
    {row['area']} m²<br><br>
    <img src="{row['image']}" width="200"><br>
    <a href="{row['url']}" target="_blank">Open listing</a>
    """

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        fill=True,
        popup=folium.Popup(popup_html, max_width=250)
    ).add_to(all_cluster)

all_layer.add_to(m)

# layer control
folium.LayerControl(collapsed=False).add_to(m)

# grouped layer control
GroupedLayerControl(
    groups={
        "Property Layers": [
            all_layer,
            sreality_layer,
            bezrealitky_layer
        ]
    },
    exclusive_groups=True,
    collapsed=False
).add_to(m)

# save
m.save("heatmap.html")